# 실습 22. 효용과 QALY 계산

사회약학 연구방법 노트의 '실습 22 효용과 QALY 계산'에 나오는 셀을 차례로 모은 노트북입니다. 22장의 가상 임상시험 자료(환자 320명, EQ-5D-5L 응답 1,593건)로 한국 가치 세트의 효용, 상태별 효용(0.780과 0.621), 환자별 24개월 QALY와 기저 효용 보정(0.117에서 0.094)을 계산하고, 20–25장 공통 예시 모형의 20년 QALY(증분 0.511)를 상태별로 나눈 뒤 할인 시점을 바꿔 봅니다. 셀을 위에서부터 차례로 실행하세요. 자료는 사이트에서 바로 읽으므로 인터넷 연결이 필요합니다. 설명과 출력 읽는 법은 사이트의 실습 22 쪽에 있습니다.


## a. 실습 데이터 준비

22장의 가상 임상시험(진행성 신세포암, 신약 A 대 표준요법 B, 군당 160명)의 EQ-5D-5L 응답과 환자별 진행·사망 시점을 읽고 확인합니다. 셀 1은 출력을 만든 패키지의 버전을 찍습니다. 내 화면의 출력이 사이트와 다를 때 먼저 비교합니다.

### 셀 1. 패키지 불러오기와 버전 확인

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy.integrate import trapezoid
import matplotlib, scipy, statsmodels

print("numpy      ", np.__version__)
print("pandas     ", pd.__version__)
print("matplotlib ", matplotlib.__version__)
print("scipy      ", scipy.__version__)
print("statsmodels", statsmodels.__version__)

### 셀 2. 설문 응답과 환자 자료 불러오기

In [ ]:
BASE = "https://socialp-ajou.tecentriq12.workers.dev/data/"
UA = {"User-Agent": "Mozilla/5.0"}   # 사이트가 파이썬 기본 요청을 막아 브라우저처럼 보이게 함
df = pd.read_csv(BASE + "lab22_eq5d.csv", storage_options=UA,
                 dtype={"eq5d": str})          # 응답 코드는 글자로 읽는다
pts = pd.read_csv(BASE + "lab22_patients.csv", storage_options=UA)
print(df.shape, pts.shape)
print(pts.head(3))
df.head()

### 셀 3. 자료 확인

In [ ]:
print("환자:", pts.groupby("arm").size().to_dict())
print("시점별 응답:", df.groupby("month").size().to_dict())
print("상태별 응답:", df["state"].value_counts().to_dict())
print("진행 뒤에 응답한 환자:", df.loc[df["state"] == "PD", "id"].nunique(), "명")

died = pts["t_death"] <= 24                     # 24개월 안에 사망
print("24개월 안의 사망:", pts[died].groupby("arm").size().to_dict())
print("진행 없이 사망(t_prog가 inf):", np.isinf(pts["t_prog"]).sum(), "명")

visits = np.array([0, 3, 6, 12, 18, 24])
alive = pts["t_death"].apply(lambda d: (visits < d).sum())   # 환자마다 살아 있던 방문 수
print("살아 있던 방문:", alive.sum(), "번, 응답:", len(df), "건")

### 셀 4. 치료 시작 시점의 세 가지 숫자

In [ ]:
b0 = df[df["month"] == 0].copy()                # 치료 시작 시점, 환자마다 한 행
dims = ["mobility", "selfcare", "activity", "pain", "anxiety"]
for i, d in enumerate(dims):
    b0[d] = b0["eq5d"].str[i].astype(int)       # 코드의 i번째 자리 = 그 영역의 수준

print((b0[dims] > 1).groupby(b0["arm"]).sum())  # 수준 2-5(문제 있음)인 환자 수
b0.groupby("arm").agg(utility=("utility", "mean"), utility_sd=("utility", "std"),
                      vas=("vas", "mean"), vas_sd=("vas", "std")).round(3)

## b. 효용 점수 구하기

한국 EQ-5D-5L 가치 세트를 함수로 짜서 응답 코드를 효용으로 바꾸고, 3L 가치 세트 두 개(한국, 영국)를 견준 뒤, 상태별 평균 효용을 환자 단위로 묶은 신뢰구간과 함께 구합니다.

### 셀 5. 한국 EQ-5D-5L 가치 세트 함수

In [ ]:
# 한국 EQ-5D-5L 가치 세트(Kim 등, 2016)의 감점. 영역마다 수준 1, 2, 3, 4, 5
KR5 = {"mobility": [0, 0.046, 0.058, 0.133, 0.251],
       "selfcare": [0, 0.032, 0.050, 0.078, 0.122],
       "activity": [0, 0.021, 0.051, 0.100, 0.175],
       "pain":     [0, 0.042, 0.053, 0.166, 0.207],
       "anxiety":  [0, 0.033, 0.046, 0.102, 0.137]}

def kr5l(code):
    """다섯 자리 응답 코드(글자) -> 한국 EQ-5D-5L 효용"""
    lv = [int(x) for x in code]              # "21222" -> [2, 1, 2, 2, 2]
    if lv == [1, 1, 1, 1, 1]:
        return 1.0                           # 완전한 건강은 감점 없음
    u = 1 - 0.096                            # 11111이 아니면 붙는 상수 감점
    for dim, level in zip(KR5, lv):
        u = u - KR5[dim][level - 1]          # 영역별 감점 (수준 1이면 0)
    if max(lv) >= 4:
        u = u - 0.078                        # 수준 4나 5가 하나라도 있으면
    return round(u, 3)

for code in ["21222", "31342", "11211", "55555", "11111"]:
    print(code, kr5l(code))

### 셀 6. 응답 1,593건을 효용으로 바꾸고 utility 열과 견주기

In [ ]:
df["u_calc"] = df["eq5d"].map(kr5l)            # 1,593건 모두
print("서로 다른 응답 코드:", df["eq5d"].nunique(), "가지")
print("utility 열과의 차이(가장 큰 값):", (df["u_calc"] - df["utility"]).abs().max())
print("11111(효용 1) 응답:", (df["eq5d"] == "11111").sum(), "건")
df[df["u_calc"] < 0][["id", "arm", "month", "state", "eq5d", "u_calc"]]

### 셀 7. EQ-5D-3L 가치 세트 두 개로 같은 응답 바꾸기

In [ ]:
# EQ-5D-3L 가치 세트: 상수 감점, 수준 3이 하나라도 있으면 붙는 감점, 영역별 (수준 2, 수준 3) 감점
KR3 = {"const": 0.050, "n3": 0.050,             # 한국 (Lee 등, 2009)
       "dec": [(0.096, 0.418), (0.046, 0.136), (0.051, 0.208),
               (0.037, 0.151), (0.043, 0.158)]}
UK3 = {"const": 0.081, "n3": 0.269,             # 영국 (Dolan, 1997)
       "dec": [(0.069, 0.314), (0.104, 0.214), (0.036, 0.094),
               (0.123, 0.386), (0.071, 0.236)]}

def eq3l(code, vs):
    """다섯 자리 3L 응답 코드 -> 가치 세트 vs의 효용"""
    lv = [int(x) for x in code]
    if lv == [1, 1, 1, 1, 1]:
        return 1.0
    u = 1 - vs["const"]
    for (d2, d3), level in zip(vs["dec"], lv):
        u = u - {1: 0, 2: d2, 3: d3}[level]
    if 3 in lv:
        u = u - vs["n3"]
    return round(u, 3)

print("검산 21232 (한국):", eq3l("21232", KR3))
codes = ["11121", "21222", "33333"]
t3 = pd.DataFrame({"korea": [eq3l(x, KR3) for x in codes],
                   "uk": [eq3l(x, UK3) for x in codes]}, index=codes)
t3["diff"] = t3["korea"] - t3["uk"]
t3.round(3)

### 셀 8. 상태별 평균 효용과 환자 단위로 묶은 신뢰구간

In [ ]:
naive = smf.ols("utility ~ 0 + state", data=df).fit()
clus = smf.ols("utility ~ 0 + state", data=df).fit(
    cov_type="cluster", cov_kwds={"groups": df["id"]})   # 환자 단위로 묶음

ci = clus.conf_int()
st = pd.DataFrame({"mean": clus.params,
                   "se_naive": naive.bse,
                   "se_cluster": clus.bse,
                   "lo": ci[0], "hi": ci[1]})
st.index = [k[6:-1] for k in st.index]         # "state[PF]" -> "PF"
st["n"] = df["state"].value_counts()
st.loc[["PF", "PD"]].round({"mean": 3, "se_naive": 4, "se_cluster": 4,
                             "lo": 3, "hi": 3})

### 셀 9. EQ VAS ÷ 100과 효용 견주기

In [ ]:
by = df.groupby("state").agg(n=("id", "size"), utility=("utility", "mean"),
                             vas=("vas", "mean"))
by["vas_100"] = by["vas"] / 100                # EQ VAS를 100으로 나눈 값
by = by.loc[["PF", "PD"]]
gap = by.loc["PF"] - by.loc["PD"]              # 무진행 - 진행
print("두 상태의 차이  효용:", round(gap["utility"], 3),
      "  VAS/100:", round(gap["vas_100"], 3))
by.round(3)

## c. QALY와 할인

사다리꼴 공식으로 환자별 24개월 QALY를 구해 두 군을 견주고(기저 효용 보정 전후), 생존자만의 평균으로 계산하면 얼마나 틀리는지 봅니다. 끝에서는 공통 예시 모형의 20년 QALY를 상태별로 나누고 할인 시점을 바꿔 봅니다.

### 셀 10. 환자 한 명의 QALY (사다리꼴 공식)

In [ ]:
def path(month, u, t_death, end=24):
    """측정 시점(개월)과 효용 -> 사망과 추적 끝을 이은 (시점, 효용) 경로"""
    t, u = list(month), list(u)
    if t_death <= end:                    # 추적 기간 안에 사망
        t, u = t + [t_death], u + [0.0]   # 사망 시점의 효용은 0
        if t_death < end:
            t, u = t + [end], u + [0.0]   # 그 뒤 추적 끝까지 0
    return np.array(t, dtype=float), np.array(u, dtype=float)

# 환자 다 (22장 표 22-4). 0, 3, 6, 12개월에 응답하고 15개월에 사망
u_c = [kr5l(x) for x in ["11221", "21222", "21232", "31241"]]
t, u = path([0, 3, 6, 12], u_c, t_death=15)
print("시점:", t)
print("효용:", u)
print("구간별 QALY:", np.round(np.diff(t) * (u[:-1] + u[1:]) / 2 / 12, 3))
print("QALY:", round(trapezoid(u, t) / 12, 3), " 생존연수:", 15 / 12)

### 셀 11. 환자 320명의 24개월 QALY와 생존연수

In [ ]:
paths, q, ly, base = {}, [], [], []
for r in pts.itertuples():                     # 환자 한 명씩
    g = df[df["id"] == r.id]                   # 이 환자의 응답 (시점 순서)
    t, u = path(g["month"], g["utility"], r.t_death)
    paths[r.id] = (t, u)                       # 뒤의 셀에서 다시 씀
    q.append(trapezoid(u, t) / 12)
    ly.append(min(r.t_death, 24) / 12)
    base.append(g["utility"].iloc[0])          # 치료 시작 시점의 효용
pts["qaly"], pts["ly"], pts["base"] = q, ly, base

print(pts.head(3).round(3))
pts.groupby("arm").agg(qaly=("qaly", "mean"), qaly_sd=("qaly", "std"),
                       ly=("ly", "mean"), base=("base", "mean")).round(3)

### 셀 12. 기저 효용 보정 전후의 QALY 차이

In [ ]:
pts["A"] = (pts["arm"] == "A").astype(int)       # 신약 A = 1, B = 0
m0 = smf.ols("qaly ~ A", data=pts).fit()
m1 = smf.ols("qaly ~ A + base", data=pts).fit()
for name, m in [("보정 전", m0), ("보정 후", m1)]:
    lo, hi = m.conf_int().loc["A"]
    print(name, " 차이", round(m.params["A"], 3),
          " 95% CI", round(lo, 3), round(hi, 3), " SE", round(m.bse["A"], 3))

mb = pts.groupby("arm")["base"].mean()
print("기저 효용의 계수:", round(m1.params["base"], 3))
print("기저 효용의 차이", round(mb["A"] - mb["B"], 4),
      "x 계수 =", round((mb["A"] - mb["B"]) * m1.params["base"], 3))

### 셀 13. 생존자만의 평균 효용으로 계산하면

In [ ]:
# 행: 시점, 열: 군. 그 시점에 살아서 답한 사람만의 평균
alive_mean = df.groupby(["month", "arm"])["utility"].mean().unstack()
alive_n = df.groupby(["month", "arm"]).size().unstack()
print(pd.concat({"mean": alive_mean.round(3), "n": alive_n}, axis=1))

wrong = trapezoid(alive_mean, visits, axis=0) / 12   # 생존자만의 평균을 이은 곡선 아래 넓이
print("생존자만의 평균 곡선:", wrong.round(3))
print("환자별 QALY의 평균  :", pts.groupby("arm")["qaly"].mean().round(3).to_numpy())

### 셀 14. 두 평균 곡선 그리기

In [ ]:
grid = np.union1d(visits, pts.loc[died, "t_death"])   # 방문 시점과 모든 사망 시점
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4), sharey=True)
for ax, arm in zip(axes, ["A", "B"]):
    ids = pts.loc[pts["arm"] == arm, "id"]
    curve = np.mean([np.interp(grid, *paths[i]) for i in ids], axis=0)
    ax.plot(grid, curve, label="All patients (dead = 0)")
    ax.plot(visits, alive_mean[arm], "o--", label="Survivors only")
    ax.set_title("Drug A" if arm == "A" else "Standard therapy B")
    ax.set_xlabel("Months since randomization")
    ax.set_xticks(visits)
    print(arm, "평균 곡선 아래 넓이:", round(trapezoid(curve, grid) / 12, 3))
axes[0].set_ylabel("Mean utility")
axes[0].set_ylim(0, 1)
axes[0].legend(fontsize=8)
plt.tight_layout()

### 셀 15. 둘째 해만 할인한 24개월 QALY

In [ ]:
def split_qaly(t, u, cut=12):
    """경로를 cut개월에서 나눠 (앞부분의 QALY, 뒷부분의 QALY)"""
    uc = np.interp(cut, t, u)                  # cut개월의 효용 (직선 위의 값)
    t1, u1 = np.append(t[t < cut], cut), np.append(u[t < cut], uc)
    t2, u2 = np.insert(t[t > cut], 0, cut), np.insert(u[t > cut], 0, uc)
    return trapezoid(u1, t1) / 12, trapezoid(u2, t2) / 12

yr = np.array([split_qaly(*paths[i]) for i in pts["id"]])   # 320행 x 2열
pts["qaly_d"] = yr[:, 0] + yr[:, 1] / 1.045               # 둘째 해만 할인
print("첫해 + 둘째 해 = 할인 전 QALY:", np.allclose(yr.sum(axis=1), pts["qaly"]))
mq = pts.groupby("arm")[["qaly", "qaly_d"]].mean()
print(mq.round(3))
print("차이  할인 전", round(mq.loc["A", "qaly"] - mq.loc["B", "qaly"], 3),
      " 둘째 해 할인", round(mq.loc["A", "qaly_d"] - mq.loc["B", "qaly_d"], 3))

### 셀 16. 공통 예시 모형을 함수 하나로

In [ ]:
p = {"pfs_med": 10, "pfs_gam": 0.95,    # 표준요법 B 무진행생존: 중앙값(개월), 모양
     "os_med": 28, "os_gam": 1.15,      # 표준요법 B 전체생존
     "hr_pfs": 0.65, "hr_os": 0.75,     # 신약 A의 위험비
     "u_pf": 0.78, "u_pd": 0.62,        # 상태별 효용 (셀 8의 0.780, 0.621)
     "du_ae_A": 0.012, "du_ae_B": 0.008,                         # 이상반응 QALY 손실
     "c_drug_A": 190, "c_drug_B": 120, "c_pf": 40, "c_pd": 250,  # 월 비용(만원)
     "c_death": 800, "c_ae_A": 120, "c_ae_B": 80,                # 1회 비용(만원)
     "disc": 0.045}                     # 연 할인율

def model(p, when="mid", months=240):
    """공통 예시의 분할생존모형 (1개월 주기, 20년). when: 할인 계수를 매기는 시점"""
    t = np.arange(months + 1)           # 주기 경계: 0, 1, ..., 240개월
    k = np.arange(months)               # 주기 번호: 0, 1, ..., 239
    years = {"mid": (k + 0.5) / 12,     # 각 달의 한가운데 (기준 분석)
             "start": k / 12,           # 각 달의 시작
             "end": (k + 1) / 12,       # 각 달의 끝
             "annual": k // 12,         # 연 단위 (첫해는 0년, 할인 없음)
             "wrong": k + 0.5}[when]    # 잘못: 개월 수를 연수 자리에
    disc = 1 / (1 + p["disc"]) ** years # 주기별 할인 계수 240개
    res = {}
    for arm in ["A", "B"]:
        hp = p["hr_pfs"] if arm == "A" else 1
        ho = p["hr_os"] if arm == "A" else 1
        s_os = np.exp(-np.log(2) * ho * (t / p["os_med"]) ** p["os_gam"])
        s_pfs = np.exp(-np.log(2) * hp * (t / p["pfs_med"]) ** p["pfs_gam"])
        s_pfs = np.minimum(s_pfs, s_os)
        pf = (s_pfs[:-1] + s_pfs[1:]) / 2                 # 반주기 보정
        pd_ = ((s_os - s_pfs)[:-1] + (s_os - s_pfs)[1:]) / 2
        died_k = s_os[:-1] - s_os[1:]                     # 주기별 사망
        r = {"pf_years": (pf * disc).sum() / 12,          # 할인한 상태별 기간(년)
             "pd_years": (pd_ * disc).sum() / 12}
        r["qaly"] = (p["u_pf"] * r["pf_years"] + p["u_pd"] * r["pd_years"]
                     - p["du_ae_" + arm])
        r["cost"] = ((pf * (p["c_drug_" + arm] + p["c_pf"]) + pd_ * p["c_pd"]
                      + died_k * p["c_death"]) * disc).sum() + p["c_ae_" + arm]
        res[arm] = r
    return res

res = model(p)
print("QALY   A", round(res["A"]["qaly"], 3), " B", round(res["B"]["qaly"], 3))
print("총비용 A", round(res["A"]["cost"], 1), " B", round(res["B"]["cost"], 1))
print("ICER", round((res["A"]["cost"] - res["B"]["cost"])
                    / (res["A"]["qaly"] - res["B"]["qaly"])))

### 셀 17. 모형의 20년 QALY를 상태별로 나누기

In [ ]:
rows = {}
for arm in ["A", "B"]:
    r = res[arm]
    rows[arm] = {"pf_years": r["pf_years"], "pd_years": r["pd_years"],
                 "life_years": r["pf_years"] + r["pd_years"],
                 "qaly_pf": p["u_pf"] * r["pf_years"],
                 "qaly_pd": p["u_pd"] * r["pd_years"],
                 "ae_loss": -p["du_ae_" + arm],
                 "qaly": r["qaly"]}
tab = pd.DataFrame(rows)
tab["diff"] = tab["A"] - tab["B"]
tab.round(3)

### 셀 18. 할인 계수를 매기는 시점 바꾸기

In [ ]:
rows = []
for when in ["mid", "start", "end", "annual", "wrong"]:
    r = model(p, when)
    dq = r["A"]["qaly"] - r["B"]["qaly"]
    dc = r["A"]["cost"] - r["B"]["cost"]
    rows.append([when, r["A"]["qaly"], r["B"]["qaly"], dq, dc / dq])
when_tab = pd.DataFrame(rows, columns=["when", "qaly_A", "qaly_B",
                                       "d_qaly", "icer"]).set_index("when")
print("12번째 달의 할인 계수  기준:", round(1 / 1.045 ** (11.5 / 12), 3),
      " wrong:", round(1 / 1.045 ** 11.5, 3))
when_tab.round(3).round({"icer": 0})

## 과제 정답

**과제 1.** 셀 8의 무진행 행(평균 0.780, 95% CI 0.765~0.794, 응답 1,184건)을 한 문장으로 옮기고, 이 숫자가 뜻하지 않는 것을 두 가지 이상 씁니다.

### 셀 19. 과제 1 정답. 무진행 상태의 효용 한 행 읽기

In [ ]:
pf_resp = df[df["state"] == "PF"]
print("응답", len(pf_resp), "건, 환자", pf_resp["id"].nunique(), "명")
print("효용이 1인 응답의 비율:", round((pf_resp["utility"] == 1).mean(), 3))
print("개별 응답의 가운데 95% 범위:",
      pf_resp["utility"].quantile([0.025, 0.975]).round(3).tolist())
print("군별 평균:", pf_resp.groupby("arm")["utility"].mean().round(3).to_dict())

**과제 2.** 마지막 측정값이 사망 직전까지 유지되고 사망 시점에 0이 된다고 보는 규칙으로 바꿔 환자 다의 QALY, 두 군의 평균 QALY, 기저 효용 보정 전후의 차이를 다시 구하고 왜 달라지는지 설명합니다.

### 셀 20. 과제 2 정답. 마지막 측정값을 사망 직전까지 유지

In [ ]:
def path_locf(month, u, t_death, end=24):
    """마지막 측정값이 사망 직전까지 유지된다고 보는 경로"""
    t, u = list(month), list(u)
    if t_death <= end:                    # 사망 직전까지 마지막 값
        t, u = t + [t_death, t_death], u + [u[-1], 0.0]
        if t_death < end:
            t, u = t + [end], u + [0.0]
    return np.array(t, dtype=float), np.array(u, dtype=float)

t, u = path_locf([0, 3, 6, 12], u_c, t_death=15)
print("환자 다:", round(trapezoid(u, t) / 12, 3))

q2 = []
for r in pts.itertuples():
    g = df[df["id"] == r.id]
    t, u = path_locf(g["month"], g["utility"], r.t_death)
    q2.append(trapezoid(u, t) / 12)
pts["qaly_locf"] = q2
print(pts.groupby("arm")[["qaly", "qaly_locf"]].mean().round(3))
for f in ["qaly_locf ~ A", "qaly_locf ~ A + base"]:
    m = smf.ols(f, data=pts).fit()
    lo, hi = m.conf_int().loc["A"]
    print(f, " 차이", round(m.params["A"], 3),
          " 95% CI", round(lo, 3), round(hi, 3))

**과제 3.** 효용 대신 EQ VAS ÷ 100으로 QALY를 계산하고 모형에도 상태별 VAS ÷ 100 평균을 넣은 동료의 분석을 재현하고, 틀린 곳 두 가지를 찾습니다.

### 셀 21. 과제 3 정답. EQ VAS ÷ 100으로 계산한 동료의 QALY

In [ ]:
# 동료의 코드: 효용 대신 EQ VAS ÷ 100
q_vas = []
for r in pts.itertuples():
    g = df[df["id"] == r.id]
    t, u = path(g["month"], g["vas"] / 100, r.t_death)
    q_vas.append(trapezoid(u, t) / 12)
pts["qaly_vas"] = q_vas
m = smf.ols("qaly_vas ~ A", data=pts).fit()
lo, hi = m.conf_int().loc["A"]
print("VAS/100 QALY:", pts.groupby("arm")["qaly_vas"].mean().round(3).to_dict(),
      " 차이", round(m.params["A"], 3), " 95% CI", round(lo, 3), round(hi, 3))

p_vas = dict(p, u_pf=0.73, u_pd=0.64)          # 셀 9의 상태별 VAS/100 평균
for name, pp in [("효용", p), ("VAS/100", p_vas)]:
    r = model(pp)
    dq = r["A"]["qaly"] - r["B"]["qaly"]
    print(name, " 증분 QALY", round(dq, 3),
          " ICER", round((r["A"]["cost"] - r["B"]["cost"]) / dq))